# Lab 01 - Classifying Spam Emails

Notebook này được tổ chức theo từng step giống notebook mẫu: load data, EDA trước balance, preprocessing/balance, EDA sau balance, feature engineering, train model, evaluate, tune/check, và deploy.

## Step 0 — Import Libraries And Setup

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

for module_path in [NOTEBOOKS_DIR, PROJECT_ROOT]:
    module_path = str(module_path)
    if module_path in sys.path:
        sys.path.remove(module_path)
    sys.path.insert(0, module_path)

import crawl
import preprocess
import model_from_scratch
import eda

crawl = importlib.reload(crawl)
preprocess = importlib.reload(preprocess)
model_from_scratch = importlib.reload(model_from_scratch)
eda = importlib.reload(eda)
model_checker = model_from_scratch.SklearnModelChecker()

paths = crawl.project_paths(PROJECT_ROOT)
print(f"crawl module: {crawl.__file__}")
print(f"preprocess module: {preprocess.__file__}")
print(f"model module: {model_from_scratch.__file__}")
print(f"eda module: {eda.__file__}")
PROJECT_ROOT

## Step 1 — Raw/Processed Data: Load Dataset

Load dữ liệu raw và dữ liệu đã xử lý/balance hiện có trong project.

In [ ]:
data, raw_data, paths = crawl.load_datasets(PROJECT_ROOT)
overview = crawl.dataset_overview(data, raw_data)

print(f"Processed rows: {overview['processed_rows']:,}")
print(f"Raw rows: {overview['raw_rows']:,}" if overview["raw_rows"] else "Raw dataset not found")
display(data.head(3))

### Step 1.1 — Label And Source Distribution

In [ ]:
display(crawl.label_counts(data))
display(crawl.source_counts(data, top_n=15))

## Step 2 — Data Processing: Clean Text

Minh hoạ cách email được làm sạch trước khi đưa vào mô hình.

In [ ]:
display(preprocess.processed_sample(data, rows=3, random_state=42))

In [ ]:
raw_example, clean_example = preprocess.example_cleaning()

print("Raw example:")
print(raw_example)
print("\nCleaned example using notebooks/preprocess.py:")
print(clean_example)

## Step 3 — Raw-To-Clean Check On Real Rows

Cell này chứng minh preprocessing chạy trên dữ liệu raw thật, không chỉ trên ví dụ tự tạo.

In [ ]:
raw_to_clean = preprocess.raw_to_clean_sample(raw_data, rows=5)
if raw_to_clean.empty:
    print("Raw dataset is not available, so this check is skipped.")
else:
    display(raw_to_clean)

## Step 4 — EDA Before Balance

Tạo bản dữ liệu đã clean và đủ điều kiện train nhưng **chưa balance**. Step này dùng để so sánh phân phối nhãn/source trước khi cân bằng dữ liệu.

In [ ]:
full_clean_data, before_balance_data, balanced_preview = preprocess.process_raw_dataset(raw_data, balance=True)

print(f"Full cleaned rows: {len(full_clean_data):,}")
print(f"Trainable rows before balance: {len(before_balance_data):,}")
print(f"Balanced preview rows: {len(balanced_preview):,}")
display(crawl.label_counts(before_balance_data))
display(eda.source_label_table(before_balance_data).head(20))
display(eda.length_summary(before_balance_data))

In [ ]:
eda.plot_eda_overview(before_balance_data, title_prefix="Before balance")

## Step 5 — EDA After Balance

Đây là EDA trên dataset cuối cùng dùng để train/test. Các biểu đồ được vẽ trực tiếp trong notebook.

In [ ]:
display(eda.source_label_table(data).head(20))
display(eda.length_summary(data))

In [ ]:
eda.plot_eda_overview(data, title_prefix="After balance")

### Step 5 Notes — EDA Reading

## Step 6 — Train/Test Split

Split có stratify theo `source + label` khi đủ dữ liệu, nếu không thì stratify theo label.

In [ ]:
split = preprocess.split_training_data(data)
TEXT_COLUMN = split["text_column"]
train_data = split["train_data"]
test_data = split["test_data"]
x_train = split["x_train"]
y_train = split["y_train"]
x_test = split["x_test"]
y_test = split["y_test"]

print(f"Text column: {TEXT_COLUMN}")
print(f"Train rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")
display(preprocess.train_source_crosstab(train_data, top_n=15))

## Step 7 — Feature Engineering: TF-IDF

In [ ]:
tfidf_shape, tfidf_features = model_checker.tfidf_feature_preview(x_train, preview_count=30)

print(f"TF-IDF matrix shape: {tfidf_shape[0]:,} emails x {tfidf_shape[1]:,} features")
print("Example features:")
print(tfidf_features)

## Step 8 — Three Algorithms From Scratch + Sklearn Checker

Tự code 3 thuật toán: Multinomial Naive Bayes, Logistic Regression, Linear SVM. Lớp `SklearnModelChecker` train các model tương ứng bằng sklearn để kiểm tra lại kết quả. Step này dùng sample train có giới hạn để notebook chạy nhanh khi báo cáo.

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd

MODEL_SAMPLE_ROWS = 3000
CHECK_SAMPLE_ROWS = 1000

model_train_frame = pd.DataFrame(
    {"text": pd.Series(x_train).reset_index(drop=True), "label": pd.Series(y_train).reset_index(drop=True)}
)
model_test_frame = pd.DataFrame(
    {"text": pd.Series(x_test).reset_index(drop=True), "label": pd.Series(y_test).reset_index(drop=True)}
)

if len(model_train_frame) > MODEL_SAMPLE_ROWS:
    model_train_frame, _ = train_test_split(
        model_train_frame,
        train_size=MODEL_SAMPLE_ROWS,
        random_state=42,
        stratify=model_train_frame["label"],
    )
if len(model_test_frame) > CHECK_SAMPLE_ROWS:
    model_test_frame, _ = train_test_split(
        model_test_frame,
        train_size=CHECK_SAMPLE_ROWS,
        random_state=42,
        stratify=model_test_frame["label"],
    )

scratch_vectorizer = model_checker.make_tfidf_vectorizer()
X_train_scratch = scratch_vectorizer.fit_transform(model_train_frame["text"])
X_test_scratch = scratch_vectorizer.transform(model_test_frame["text"])
y_train_scratch = model_train_frame["label"].to_numpy()
y_test_scratch = model_test_frame["label"].to_numpy()

scratch_models = {
    "Multinomial Naive Bayes": model_from_scratch.ScratchMultinomialNB(alpha=1.0),
    "Logistic Regression": model_from_scratch.ScratchLogisticRegression(),
    "Linear SVM": model_from_scratch.ScratchLinearSVM(),
}

sklearn_checker = model_from_scratch.SklearnModelChecker().fit(X_train_scratch, y_train_scratch)
model_rows = []
for model_name, scratch_model in scratch_models.items():
    scratch_model.fit(X_train_scratch, y_train_scratch)
    scratch_predictions = scratch_model.predict(X_test_scratch)
    sklearn_predictions = sklearn_checker.predict(model_name, X_test_scratch)

    model_rows.append(
        {
            "algorithm": model_name,
            "implementation": "from scratch",
            "accuracy": accuracy_score(y_test_scratch, scratch_predictions),
            "agreement_with_sklearn": (scratch_predictions == sklearn_predictions).mean(),
        }
    )
    model_rows.append(
        {
            "algorithm": model_name,
            "implementation": "sklearn checker",
            "accuracy": accuracy_score(y_test_scratch, sklearn_predictions),
            "agreement_with_sklearn": 1.0,
        }
    )

display(pd.DataFrame(model_rows))

## Step 9 — Model Training

In [ ]:
model, predictions, baseline_accuracy, model_accuracy = model_checker.train_project_model(x_train, y_train, x_test, y_test)

print(f"Baseline accuracy: {baseline_accuracy:.4f}")
print(f"Naive Bayes accuracy: {model_accuracy:.4f}")

## Step 10 — Evaluate Model

In [ ]:
print(eda.classification_report_text(y_test, predictions))

ConfusionMatrixDisplay.from_predictions(y_test, predictions)
plt.title("TF-IDF + Multinomial Naive Bayes")
plt.tight_layout()
plt.show()

### Step 10 Notes — Reading The Metrics

In [ ]:
display(eda.top_tokens(model, top_n=15))

In [ ]:
display(eda.per_source_scores(test_data, predictions).head(15))

## Step 11 — Cross-Source Holdout Check

In [ ]:
display(eda.cross_source_holdout(data, TEXT_COLUMN, paths["metrics_dir"]).head(15))

## Step 12 — Optional Model Comparison

In [ ]:
display(model_checker.compare_models(data, TEXT_COLUMN, sample_rows=50_000))

## Step 13 — Save And Reuse The Model

In [ ]:
saved_model_path = model_checker.save_model(model, paths["model"])
print(f"Saved model to {saved_model_path.relative_to(PROJECT_ROOT)}")

In [ ]:
display(model_checker.predict_new_emails(paths["model"]))

## Step 14 — Project Reports

In [ ]:
display(crawl.report_status(paths["metrics_dir"], PROJECT_ROOT))

print("\nSaved classification report:\n")
print(crawl.read_classification_report(paths["metrics_dir"]))